# DeepInspire evaluation

Reproduces the evaluation of DeepInspire on the public data set (Zenodo, https://doi.org/10.5281/zenodo.24286276):

1. **Prediction.** Runs the lung and rib MA-Net models and the ResNet34 classifier on the training and test images and measures the rib-over-lung (ROL) value, the segmentation accuracy and, for the test images, the agreement with the two observers' full-resolution masks.
2. **Statistics.** Fits the ROL cutoff on the training set (Youden index) and reports test performance with 95% CIs (stratified bootstrap of images, 10,000 resamples, plus a patient-level bootstrap), the paired DeLong and McNemar comparisons with ResNet34, the nine readers and their ICC, segmentation errors, alternative cutoffs, the full-resolution check and Table 1.
3. **Table 2.** DSC per side for the automated segmentation and both annotators.

All results are written to `OUT_DIR`. Figures are made in `visualization.ipynb`. Statistics are implemented in NumPy with fixed random seeds, so the numbers are identical between runs.

In [1]:
import os, re, math, glob, time, collections
import numpy as np, pandas as pd, cv2, torch, tifffile
import segmentation_models_pytorch as smp
from torch import nn
from torchvision import models

# Paths relative to this notebook (DeepInspire/Notebook/). Unzip Data.zip and Model.zip
# from https://doi.org/10.5281/zenodo.24286276 next to the repository.
DATA_DIR = '../../Data'
MODEL_DIR = '../../Model'
OUT_DIR = '../../results'
os.makedirs(OUT_DIR, exist_ok=True)

/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <52D1D93C-FC3D-335D-8475-03482908DF9F> /opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <ECC148AF-20FF-3EEE-BC75-4DD3E7455393> /opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


## 1. Prediction

Load the three trained models.

In [2]:
PIX_MM = 0.143            # native pixel size reported in the manuscript
torch.set_grad_enabled(False)
device = torch.device('cpu')


class MAnetWithDropout(nn.Module):  # copied from evaluation.ipynb
    def __init__(self, dropout_p=0.2, **kw):
        super().__init__()
        self.model = smp.MAnet(**kw)
        self.dropout = nn.Dropout2d(p=dropout_p)

    def forward(self, x):
        f = self.model.encoder(x)
        d = self.dropout(self.model.decoder(*f))
        return self.model.segmentation_head(d)


def load_seg(path):
    m = MAnetWithDropout(dropout_p=0.4, encoder_name='resnet34', encoder_weights=None,
                         in_channels=3, classes=3)
    m.load_state_dict(torch.load(path, map_location=device)['model'])
    return m.eval()


rib_model = load_seg(os.path.join(MODEL_DIR, 'rib_MAnet_20260209_034825_best.pth'))
lung_model = load_seg(os.path.join(MODEL_DIR, 'lung_MAnet_20260209_042722_best.pth'))
clf = models.resnet34(weights=None)
clf.fc = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(clf.fc.in_features, 2))
clf.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'Classification_resnet34_20260209_051509_best.pth'),
                               map_location=device)['model'])
clf.eval()
n_params = {'MAnet_ResNet34 (each segmentation model)': sum(p.numel() for p in rib_model.parameters()),
            'ResNet34 classifier': sum(p.numel() for p in clf.parameters())}
pd.Series(n_params).to_csv(os.path.join(OUT_DIR, 'model_parameters.csv'), header=['parameters'])
print(n_params)

/var/folders/g5/zsclc1cn68nbqrd2tbc037br0000gn/T/ipykernel_74849/1308848281.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(path, map_locat

{'MAnet_ResNet34 (each segmentation model)': 31783923, 'ResNet34 classifier': 21285698}


/var/folders/g5/zsclc1cn68nbqrd2tbc037br0000gn/T/ipykernel_74849/1308848281.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clf.load_state_dict(torch.load(os.path.join(

Preprocessing, ROL and segmentation metrics.

ROL (%) is the percentage of 9th-posterior-rib pixels that lie inside the lung fields (both sides combined). HD95 and ASSD are computed in mm from the native pixel size.

In [3]:
prep = smp.encoders.get_preprocessing_fn('resnet34', 'imagenet')


def read_img(path):
    raw = cv2.imread(path, 1)
    h, w = raw.shape[:2]
    img = cv2.resize(raw, (512, 512), interpolation=cv2.INTER_NEAREST)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
    return img, (h, w)


def seg_logits(model, img):
    x = prep(img.copy()).astype(np.float64).transpose(2, 0, 1)[None]
    return model(torch.from_numpy(x).float())[0]          # (3,512,512)


def clf_prob(img):
    x = torch.from_numpy(img.transpose(2, 0, 1)[None].copy()).float()
    return torch.softmax(clf(x), 1)[0, 1].item()


def rol(rib, lung):
    """ROL (%) = rib pixels inside lung / all rib pixels; both sides combined."""
    r, l = rib > 0, lung > 0
    return 100.0 * np.logical_and(r, l).sum() / r.sum() if r.sum() else 0.0


def dsc(a, b):
    a, b = a.astype(bool), b.astype(bool)
    s = a.sum() + b.sum()
    return 1.0 if s == 0 else 2.0 * np.logical_and(a, b).sum() / s


def boundary_pts(m, sx, sy):
    m = m.astype(np.uint8)
    edge = m - cv2.erode(m, np.ones((3, 3), np.uint8))
    yx = np.argwhere(edge > 0).astype(np.float32)
    return torch.from_numpy(yx * np.array([sy, sx], np.float32))


def hd95_assd(a, b, sx, sy):
    """Symmetric HD95 and ASSD in mm between binary masks on a 512 grid with
    (possibly anisotropic) pixel size sx, sy."""
    pa, pb = boundary_pts(a, sx, sy), boundary_pts(b, sx, sy)
    if len(pa) == 0 or len(pb) == 0:
        return np.nan, np.nan
    d = torch.cdist(pa, pb)
    dab, dba = d.min(1).values.numpy(), d.min(0).values.numpy()
    allD = np.concatenate([dab, dba])
    return float(np.percentile(allD, 95)), float(allD.mean())


def labels_from_gt(path):
    m = tifffile.imread(path)
    return np.argmax(m, axis=-1) if m.ndim == 3 else m


def num(fn):
    return re.findall(r'(\d{3})\.(?:png|tiff)$', fn)[0]

Run the models on every training and test image (about 1 s per image on a CPU). For the test images, the predicted logits are also upsampled (bilinear) to the original image size and compared with the observers' full-resolution masks.

In [4]:
rows = []
for split in ['train', 'test']:
    for cls, folder, y in [('Adequate', 'full', 0), ('Inadequate', 'not full', 1)]:
        for ip in sorted(glob.glob(os.path.join(DATA_DIR, split, folder, 'image', '*.png'))):
            t0 = time.time()
            fn = os.path.basename(ip)
            img, (h, w) = read_img(ip)
            rl, ll = seg_logits(rib_model, img), seg_logits(lung_model, img)
            pr, pl = rl.argmax(0).numpy(), ll.argmax(0).numpy()
            gr = labels_from_gt(os.path.join(DATA_DIR, split, folder, 'mask', 'rib', 'rib_' + fn.replace('.png', '.tiff')))
            gl = labels_from_gt(os.path.join(DATA_DIR, split, folder, 'mask', 'lung', 'lung_' + fn.replace('.png', '.tiff')))
            sx, sy = w * PIX_MM / 512, h * PIX_MM / 512
            r = dict(split=split, image=fn, number=num(fn), y_inadequate=y, native_w=w, native_h=h,
                     rol_pred=rol(pr, pl), rol_gt=rol(gr, gl), resnet_prob=clf_prob(img))
            for name, p, g in [('rib', pr, gr), ('lung', pl, gl)]:
                r[f'dsc_{name}_left'] = dsc(p == 1, g == 1)
                r[f'dsc_{name}_right'] = dsc(p == 2, g == 2)
                r[f'dsc_{name}_both'] = dsc(p > 0, g > 0)
                r[f'hd95_{name}_mm'], r[f'assd_{name}_mm'] = hd95_assd(p > 0, g > 0, sx, sy)
                r[f'{name}_pred_empty'] = int((p > 0).sum() == 0)
            if split == 'test':
                # native-resolution check: upsample logits to the native size
                up = lambda L: torch.nn.functional.interpolate(L[None], size=(h, w), mode='bilinear',
                                                                align_corners=False)[0].argmax(0).numpy()
                nr, nl = up(rl), up(ll)
                r['rol_pred_native'] = rol(nr, nl)
                for o in ['observer1', 'observer2']:
                    orib = tifffile.imread(os.path.join(DATA_DIR, 'human_segment', f'mask by {o}', 'Rib', r['number'] + '.tiff'))
                    olung = tifffile.imread(os.path.join(DATA_DIR, 'human_segment', f'mask by {o}', 'Lung', r['number'] + '.tiff'))
                    r[f'{o}_shape_ok'] = int(orib.shape[:2] == (h, w) and olung.shape[:2] == (h, w))
                    r[f'rol_{o}_native'] = rol(orib, olung)
                    o512r = cv2.resize(orib.astype(np.uint8), (512, 512), interpolation=cv2.INTER_NEAREST)
                    o512l = cv2.resize(olung.astype(np.uint8), (512, 512), interpolation=cv2.INTER_NEAREST)
                    r[f'rol_{o}_512'] = rol(o512r, o512l)
                    if r[f'{o}_shape_ok']:
                        r[f'dsc_rib_pred_vs_{o}_native'] = dsc(nr > 0, orib > 0)
                        r[f'dsc_lung_pred_vs_{o}_native'] = dsc(nl > 0, olung > 0)
                    r[f'dsc_rib_pred_vs_{o}_512'] = dsc(pr > 0, o512r > 0)
                    r[f'dsc_lung_pred_vs_{o}_512'] = dsc(pl > 0, o512l > 0)
                    r[f'dsc_rib_gt_vs_{o}_512'] = dsc(gr > 0, o512r > 0)
            r['seconds'] = time.time() - t0
            rows.append(r)

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, 'per_image_results.csv'), index=False)
print('per_image_results.csv:', len(df), 'images', df.groupby('split').size().to_dict())

per_image_results.csv: 199 images {'test': 51, 'train': 148}


## 2. Statistics

Helper functions (AUC, DeLong test, metrics, bootstrap, McNemar test, Youden cutoff, ICC).

In [5]:
A = O = OUT_DIR  # step 1 output is read from, and results are written to, OUT_DIR
rng = np.random.default_rng(42)
B = 10000
out_lines = []


def say(*a):
    s = ' '.join(str(x) for x in a)
    print(s)
    out_lines.append(s)


# ---------------------------------------------------------------- basic stats
def auc(y, s):
    """AUC = P(score_pos > score_neg) + 0.5 P(tie)."""
    y, s = np.asarray(y), np.asarray(s, float)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    gt = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return gt / (len(pos) * len(neg))


def delong(y, s1, s2):
    """Paired DeLong test for two correlated AUCs (DeLong 1988)."""
    y = np.asarray(y)
    def comps(s):
        s = np.asarray(s, float); pos, neg = s[y == 1], s[y == 0]
        psi = (pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :])
        return psi.mean(1), psi.mean(0), psi.mean()
    v10a, v01a, a1 = comps(s1); v10b, v01b, a2 = comps(s2)
    m, n = len(v10a), len(v01a)
    S10 = np.cov(np.vstack([v10a, v10b])); S01 = np.cov(np.vstack([v01a, v01b]))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    z = (a1 - a2) / math.sqrt(var) if var > 0 else 0.0
    return a1, a2, z, math.erfc(abs(z) / math.sqrt(2))


def metrics(y, pred):
    y, p = np.asarray(y), np.asarray(pred)
    tp = int(((p == 1) & (y == 1)).sum()); tn = int(((p == 0) & (y == 0)).sum())
    fp = int(((p == 1) & (y == 0)).sum()); fn = int(((p == 0) & (y == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    acc = (tp + tn) / len(y)
    f1 = 2 * prec * sens / (prec + sens) if (prec + sens) and not np.isnan(prec) else np.nan
    bacc = (sens + spec) / 2
    return dict(Sensitivity=sens, Specificity=spec, Precision=prec, Accuracy=acc, F1=f1, **{'Balanced accuracy': bacc})


MNAMES = ['Sensitivity', 'Specificity', 'Precision', 'Accuracy', 'Balanced accuracy', 'F1']


def strat_idx(y, g=None):
    g = rng if g is None else g
    y = np.asarray(y); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    return np.concatenate([g.choice(i1, len(i1)), g.choice(i0, len(i0))])


def cluster_idx(clusters):
    u = np.unique(clusters); pick = rng.choice(u, len(u))
    pos = collections.defaultdict(list)
    for i, c in enumerate(clusters): pos[c].append(i)
    return np.concatenate([pos[c] for c in pick])


def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)


def fmt(v, lo, hi):
    return f'{v:.3f} ({lo:.3f}–{hi:.3f})'


def mcnemar_exact(y, p1, p2):
    c1, c2 = np.asarray(p1) == y, np.asarray(p2) == y
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    n, k = b + c, min(b, c)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return b, c, p


def youden_cutoff(rol, y):
    """Smallest ROL value maximising J when 'inadequate' = ROL <= cutoff
    (same rule as sklearn roc_curve on -ROL used in evaluation.ipynb)."""
    best, bc = -1, None
    for c in np.unique(rol):
        m = metrics(y, (rol <= c).astype(int)); j = m['Sensitivity'] + m['Specificity'] - 1
        if j > best + 1e-12: best, bc = j, c
    return bc, best


def icc2(X):
    """Shrout & Fleiss ICC(2,1) and ICC(2,k), two-way random, absolute agreement."""
    n, k = X.shape; gm = X.mean()
    msr = k * ((X.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((X.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((X - X.mean(1, keepdims=True) - X.mean(0, keepdims=True) + gm) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    i21 = (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)
    i2k = (msr - mse) / (msr + (msc - mse) / n)
    return i21, i2k


# ---------------------------------------------------------------- basic stats
def auc(y, s):
    """AUC = P(score_pos > score_neg) + 0.5 P(tie)."""
    y, s = np.asarray(y), np.asarray(s, float)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    gt = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return gt / (len(pos) * len(neg))


def delong(y, s1, s2):
    """Paired DeLong test for two correlated AUCs (DeLong 1988)."""
    y = np.asarray(y)
    def comps(s):
        s = np.asarray(s, float); pos, neg = s[y == 1], s[y == 0]
        psi = (pos[:, None] > neg[None, :]) + 0.5 * (pos[:, None] == neg[None, :])
        return psi.mean(1), psi.mean(0), psi.mean()
    v10a, v01a, a1 = comps(s1); v10b, v01b, a2 = comps(s2)
    m, n = len(v10a), len(v01a)
    S10 = np.cov(np.vstack([v10a, v10b])); S01 = np.cov(np.vstack([v01a, v01b]))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    z = (a1 - a2) / math.sqrt(var) if var > 0 else 0.0
    return a1, a2, z, math.erfc(abs(z) / math.sqrt(2))


def metrics(y, pred):
    y, p = np.asarray(y), np.asarray(pred)
    tp = int(((p == 1) & (y == 1)).sum()); tn = int(((p == 0) & (y == 0)).sum())
    fp = int(((p == 1) & (y == 0)).sum()); fn = int(((p == 0) & (y == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    acc = (tp + tn) / len(y)
    f1 = 2 * prec * sens / (prec + sens) if (prec + sens) and not np.isnan(prec) else np.nan
    bacc = (sens + spec) / 2
    return dict(Sensitivity=sens, Specificity=spec, Precision=prec, Accuracy=acc, F1=f1, **{'Balanced accuracy': bacc})


MNAMES = ['Sensitivity', 'Specificity', 'Precision', 'Accuracy', 'Balanced accuracy', 'F1']


def strat_idx(y, g=None):
    g = rng if g is None else g
    y = np.asarray(y); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    return np.concatenate([g.choice(i1, len(i1)), g.choice(i0, len(i0))])


def cluster_idx(clusters):
    u = np.unique(clusters); pick = rng.choice(u, len(u))
    pos = collections.defaultdict(list)
    for i, c in enumerate(clusters): pos[c].append(i)
    return np.concatenate([pos[c] for c in pick])


def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)


def fmt(v, lo, hi):
    return f'{v:.3f} ({lo:.3f}–{hi:.3f})'


def mcnemar_exact(y, p1, p2):
    c1, c2 = np.asarray(p1) == y, np.asarray(p2) == y
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    n, k = b + c, min(b, c)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return b, c, p


def youden_cutoff(rol, y):
    """Smallest ROL value maximising J when 'inadequate' = ROL <= cutoff
    (same rule as sklearn roc_curve on -ROL used in evaluation.ipynb)."""
    best, bc = -1, None
    for c in np.unique(rol):
        m = metrics(y, (rol <= c).astype(int)); j = m['Sensitivity'] + m['Specificity'] - 1
        if j > best + 1e-12: best, bc = j, c
    return bc, best


def icc2(X):
    """Shrout & Fleiss ICC(2,1) and ICC(2,k), two-way random, absolute agreement."""
    n, k = X.shape; gm = X.mean()
    msr = k * ((X.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((X.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((X - X.mean(1, keepdims=True) - X.mean(0, keepdims=True) + gm) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    i21 = (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)
    i2k = (msr - mse) / (msr + (msc - mse) / n)
    return i21, i2k

Data and subjects. Test images from subjects who also contributed training or validation images are excluded (none in the public data set, where they have already been removed).

In [6]:
# ---------------------------------------------------------------- data
meta = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'))
subject = dict(zip(meta.image, meta.subject_id))
subjects_by_set = meta.groupby('set').subject_id.agg(set).to_dict()
df = pd.read_csv(os.path.join(A, 'per_image_results.csv'), dtype={'number': str})
tr, te = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
lab = dict(zip(meta.image, (meta.label == 'inadequate').astype(int)))
assert all(lab[i] == v for i, v in zip(df.image, df.y_inadequate)), 'metadata label differs from image folder'
te['subject'] = te.image.map(subject)
te['subject_in_train'] = te.subject.isin(subjects_by_set.get('train', set()))
te['subject_in_valid'] = te.subject.isin(subjects_by_set.get('validation', set()))
y_tr, y = tr.y_inadequate.values, te.y_inadequate.values
say('# DeepInspire evaluation')
say(f'train n={len(tr)} (inadequate {y_tr.sum()}), test n={len(te)} (inadequate {y.sum()})')
say(f'test subjects: {te.subject.nunique()} for {len(te)} images; test images sharing a subject with train: '
    f'{te.subject_in_train.sum()}, with validation: {te.subject_in_valid.sum()}')

# sanity check against the saved notebook output

# DeepInspire evaluation
train n=148 (inadequate 74), test n=51 (inadequate 18)
test subjects: 40 for 51 images; test images sharing a subject with train: 0, with validation: 0


ROL cutoff on the training set.

In [7]:
# ---------------------------------------------------------------- cutoff
cut, jmax = youden_cutoff(tr.rol_pred.values, y_tr)
say('\n## ROL cutoff (training set, predicted masks)')
say(f'cutoff = {cut:.3f}%  (Jmax = {jmax:.4f}); training AUC = {auc(y_tr, -tr.rol_pred):.4f}')
bc = []
# B = 10,000 resamples. The first 5,000 come from the main generator, as in the first run, so that
# the test-set bootstrap draws further down stay unchanged; the other 5,000 come from a separate generator.
rng_cut = np.random.default_rng(4242)
for k in range(B):
    i = strat_idx(y_tr, None if k < 5000 else rng_cut); bc.append(youden_cutoff(tr.rol_pred.values[i], y_tr[i])[0])
lo, hi = ci(bc)
say(f'bootstrap 95% CI of cutoff ({B} stratified resamples of training images): {lo:.2f}–{hi:.2f}%')
say(f'interpretation: at the cutoff, {100 - cut:.1f}% of the ninth-posterior-rib area lies outside the lung field')
te_rol = te[~(te.subject_in_train | te.subject_in_valid)]
for name, d in [('train', tr), (f'test (n={len(te_rol)})', te_rol)]:
    for cls, lab in [(0, 'adequate'), (1, 'inadequate')]:
        v = d.rol_pred[d.y_inadequate == cls]
        say(f'ROL {name} {lab}: median {v.median():.1f} (IQR {v.quantile(.25):.1f}–{v.quantile(.75):.1f}), range {v.min():.1f}–{v.max():.1f}')
cut_gt, j_gt = youden_cutoff(tr.rol_gt.values, y_tr)
say(f'cutoff refitted on training GT-mask ROL: {cut_gt:.3f}% (J={j_gt:.4f})')


## ROL cutoff (training set, predicted masks)
cutoff = 83.918%  (Jmax = 0.9865); training AUC = 0.9938
bootstrap 95% CI of cutoff (10000 stratified resamples of training images): 82.30–83.92%
interpretation: at the cutoff, 16.1% of the ninth-posterior-rib area lies outside the lung field
ROL train adequate: median 94.3 (IQR 93.1–95.2), range 88.6–97.1
ROL train inadequate: median 60.1 (IQR 50.1–71.1), range 16.1–94.0
ROL test (n=51) adequate: median 95.2 (IQR 94.5–96.5), range 76.4–97.9
ROL test (n=51) inadequate: median 60.6 (IQR 54.0–74.9), range 35.1–86.9
cutoff refitted on training GT-mask ROL: 82.787% (J=0.9730)


Predictions at the cutoff, and the readers' answers.

In [8]:
# ---------------------------------------------------------------- predictions
te['pred_di'] = (te.rol_pred <= cut).astype(int)
te['pred_resnet'] = (te.resnet_prob >= 0.5).astype(int)
te['pred_gt_rol'] = (te.rol_gt <= cut).astype(int)
for o in ['observer1', 'observer2']:
    te[f'pred_{o}'] = (te[f'rol_{o}_native'] <= cut).astype(int)
tcols = [f'reader_{i}' for i in range(1, 10)]
te = te.merge(pd.read_csv(os.path.join(DATA_DIR, 'reader_results.csv')), on='image', how='left')
assert te[tcols].notna().all().all(), 'reader_results.csv does not cover every test image'
te_all = te.copy()
te = te[~(te.subject_in_train | te.subject_in_valid)].reset_index(drop=True); y = te.y_inadequate.values
say(f'\nTEST SET = {len(te)} images from subjects not in train or validation (inadequate {y.sum()}, subjects {te.subject.nunique()})')
R = te[tcols].values


TEST SET = 51 images from subjects not in train or validation (inadequate 18, subjects 40)


Test performance with 95% CIs, and the paired comparison with ResNet34.

In [9]:
# ---------------------------------------------------------------- metrics with CIs
say('\n## Test performance with 95% CI (stratified bootstrap, image level, B=10000)')
methods = {'DeepInspire': 'pred_di', 'ResNet34': 'pred_resnet',
           'ROL from GT masks (same cutoff)': 'pred_gt_rol',
           'ROL from observer 1 masks (criterion-guided human)': 'pred_observer1',
           'ROL from observer 2 masks (criterion-guided human)': 'pred_observer2'}
scores = {'DeepInspire': -te.rol_pred.values, 'ResNet34': te.resnet_prob.values,
          'ROL from GT masks (same cutoff)': -te.rol_gt.values,
          'ROL from observer 1 masks (criterion-guided human)': -te.rol_observer1_native.values,
          'ROL from observer 2 masks (criterion-guided human)': -te.rol_observer2_native.values}
boot_idx = [strat_idx(y) for _ in range(B)]
clusters = te.subject.fillna(te.image).values
clus_idx = [cluster_idx(clusters) for _ in range(B)]
table = []
for name, col in methods.items():
    p = te[col].values; point = metrics(y, p); point['AUC'] = auc(y, scores[name])
    bs = collections.defaultdict(list); bsc = collections.defaultdict(list)
    for i, j in zip(boot_idx, clus_idx):
        for k, v in metrics(y[i], p[i]).items(): bs[k].append(v)
        bs['AUC'].append(auc(y[i], scores[name][i]))
        bsc['AUC'].append(auc(y[j], scores[name][j])); bsc['Accuracy'].append(metrics(y[j], p[j])['Accuracy'])
    row = {'Method': name}
    for k in MNAMES + ['AUC']:
        row[k] = fmt(point[k], *ci(bs[k]))
    row['AUC (patient-level bootstrap)'] = fmt(point['AUC'], *ci(bsc['AUC']))
    row['Accuracy (patient-level bootstrap)'] = fmt(point['Accuracy'], *ci(bsc['Accuracy']))
    table.append(row)
# readers: mean over 9 readers, CI by resampling images
rm = {k: np.mean([metrics(y, R[:, r])[k] for r in range(9)]) for k in MNAMES}
rb = collections.defaultdict(list)
for i in boot_idx:
    for k in MNAMES: rb[k].append(np.nanmean([metrics(y[i], R[i, r])[k] for r in range(9)]))
table.append({'Method': 'Readers, mean of 9 (no criteria)', **{k: fmt(rm[k], *ci(rb[k])) for k in MNAMES}})
T = pd.DataFrame(table); T.to_csv(os.path.join(O, 'table_test_metrics_ci.csv'), index=False)
say(T.to_string(index=False))

say(f'\n## Paired comparison DeepInspire vs ResNet34 (same {len(te)} test images)')
a1, a2, z, pd_ = delong(y, -te.rol_pred.values, te.resnet_prob.values)
say(f'DeLong: AUC {a1:.3f} vs {a2:.3f}, z = {z:.2f}, p = {pd_:.3f}')
b_, c_, pm = mcnemar_exact(y, te.pred_di.values, te.pred_resnet.values)
say(f'McNemar exact (accuracy): DeepInspire-only correct = {b_}, ResNet-only correct = {c_}, p = {pm:.3f}')
pd1, pd2 = te.pred_di.values, te.pred_resnet.values
drows = []
for k in MNAMES + ['AUC']:
    if k == 'AUC':
        d0 = auc(y, -te.rol_pred) - auc(y, te.resnet_prob)
        ds_ = [auc(y[i], -te.rol_pred.values[i]) - auc(y[i], te.resnet_prob.values[i]) for i in boot_idx]
    else:
        d0 = metrics(y, pd1)[k] - metrics(y, pd2)[k]
        ds_ = [metrics(y[i], pd1[i])[k] - metrics(y[i], pd2[i])[k] for i in boot_idx]
    ds_ = np.asarray(ds_, float); ds_ = ds_[~np.isnan(ds_)]
    pboot = min(1.0, 2 * min((ds_ <= 0).mean(), (ds_ >= 0).mean()))
    lo, hi = ci(ds_); drows.append({'Metric': k, 'Difference (DI - ResNet)': f'{d0:+.3f}', '95% CI': f'{lo:+.3f} to {hi:+.3f}', 'bootstrap p': f'{pboot:.3f}'})
D = pd.DataFrame(drows); D.to_csv(os.path.join(O, 'table_paired_di_vs_resnet.csv'), index=False)
say(D.to_string(index=False))


## Test performance with 95% CI (stratified bootstrap, image level, B=10000)
                                            Method         Sensitivity         Specificity           Precision            Accuracy   Balanced accuracy                  F1                 AUC AUC (patient-level bootstrap) Accuracy (patient-level bootstrap)
                                       DeepInspire 0.889 (0.722–1.000) 0.970 (0.909–1.000) 0.941 (0.824–1.000) 0.941 (0.863–1.000) 0.929 (0.843–1.000) 0.914 (0.800–1.000) 0.992 (0.970–1.000)           0.992 (0.969–1.000)                0.941 (0.870–1.000)
                                          ResNet34 0.889 (0.722–1.000) 0.848 (0.727–0.970) 0.762 (0.625–0.938) 0.863 (0.765–0.941) 0.869 (0.768–0.955) 0.821 (0.700–0.923) 0.928 (0.847–0.987)           0.928 (0.834–0.987)                0.863 (0.760–0.945)
                   ROL from GT masks (same cutoff) 1.000 (1.000–1.000) 1.000 (1.000–1.000) 1.000 (1.000–1.000) 1.000 (1.000–1.000) 1.000 (1.000–1.000) 1.0

Nine radiological technologists without predefined criteria, and their agreement (ICC).

In [10]:
# ---------------------------------------------------------------- readers
say('\n## Readers (no predefined criteria)')
rows = []
for r, c in enumerate(tcols):
    m = metrics(y, R[:, r]); rows.append({'Reader': f'#{r + 1}', **{k: round(m[k], 3) for k in MNAMES}})
RT = pd.DataFrame(rows)
RT.loc[len(RT)] = {'Reader': 'Mean (SD)', **{k: f"{RT[k].mean():.3f} ({RT[k].std():.3f})" for k in MNAMES}}
RT.to_csv(os.path.join(O, 'table_readers.csv'), index=False)
say(RT.to_string(index=False))
i21, i2k = icc2(R.astype(float))
bi = [icc2(R[i].astype(float)) for i in boot_idx]
say(f'ICC(2,1) single reader = {i21:.3f} (95% CI {ci([b[0] for b in bi])[0]:.3f}–{ci([b[0] for b in bi])[1]:.3f}); '
    f'ICC(2,k=9) mean of readers = {i2k:.3f} (95% CI {ci([b[1] for b in bi])[0]:.3f}–{ci([b[1] for b in bi])[1]:.3f})')
for r in range(9):
    b_, c_, pm = mcnemar_exact(y, te.pred_di.values, R[:, r])
    say(f'  DeepInspire vs reader #{r + 1}: McNemar p = {pm:.4f} (DI-only correct {b_}, reader-only correct {c_})')
params = pd.read_csv(os.path.join(A, 'model_parameters.csv'), index_col=0)  # written by step 1
say('model parameters: ' + '; '.join(f'{i} = {v / 1e6:.2f} M' for i, v in params.parameters.items()))


## Readers (no predefined criteria)
   Reader   Sensitivity   Specificity     Precision      Accuracy Balanced accuracy            F1
       #1         0.722           1.0           1.0         0.902             0.861         0.839
       #2         0.944         0.879          0.81         0.902             0.912         0.872
       #3         0.778           1.0           1.0         0.922             0.889         0.875
       #4         0.778           1.0           1.0         0.922             0.889         0.875
       #5         0.611          0.97         0.917         0.843              0.79         0.733
       #6         0.889          0.97         0.941         0.941             0.929         0.914
       #7         0.222          0.97           0.8         0.706             0.596         0.348
       #8         0.333           1.0           1.0         0.765             0.667           0.5
       #9         0.722          0.97         0.929         0.882             0.8

Sensitivity analysis for test images from subjects in the training or validation sets (runs only if such images are present).

In [11]:
# ---------------------------------------------------------------- shared subjects
leak = (te_all.subject_in_train | te_all.subject_in_valid).values
if leak.any():  # only when the excluded test images are present (not in the public data set)
    say('\n## Sensitivity analysis: test images from subjects in train or validation')
    for name, mask in [(f'all {len(te_all)} test images', np.ones(len(te_all), bool)),
                       ('test images from subjects not in train', ~te_all.subject_in_train.values),
                       ('test images from subjects not in train or validation', ~leak),
                       ('test images from subjects in train', te_all.subject_in_train.values)]:
        yy = te_all.y_inadequate.values[mask]
        m1, m2 = metrics(yy, te_all.pred_di.values[mask]), metrics(yy, te_all.pred_resnet.values[mask])
        say(f'{name}: n={mask.sum()} (inadequate {yy.sum()}) | DeepInspire acc {m1["Accuracy"]:.3f} sens {m1["Sensitivity"]:.3f} '
            f'spec {m1["Specificity"]:.3f} F1 {m1["F1"]:.3f} AUC {auc(yy, -te_all.rol_pred.values[mask]):.3f} | '
            f'ResNet acc {m2["Accuracy"]:.3f} sens {m2["Sensitivity"]:.3f} spec {m2["Specificity"]:.3f} '
            f'AUC {auc(yy, te_all.resnet_prob.values[mask]):.3f}')

Segmentation accuracy and rib-segmentation failures.

In [12]:
# ---------------------------------------------------------------- segmentation
say('\n## Segmentation (test, predicted vs GT at 512; distances in mm)')
for s in ['lung', 'rib']:
    say(f"{s}: DSC both {te[f'dsc_{s}_both'].mean():.3f}±{te[f'dsc_{s}_both'].std():.3f}, left {te[f'dsc_{s}_left'].mean():.3f}, "
        f"right {te[f'dsc_{s}_right'].mean():.3f} | HD95 median {te[f'hd95_{s}_mm'].median():.1f} mm "
        f"(IQR {te[f'hd95_{s}_mm'].quantile(.25):.1f}–{te[f'hd95_{s}_mm'].quantile(.75):.1f}) | "
        f"ASSD median {te[f'assd_{s}_mm'].median():.1f} mm (IQR {te[f'assd_{s}_mm'].quantile(.25):.1f}–{te[f'assd_{s}_mm'].quantile(.75):.1f}) | "
        f"empty predictions {int(te[f'{s}_pred_empty'].sum())}")
for thr in [0.3, 0.5]:
    side_fail = (te.dsc_rib_left < thr) | (te.dsc_rib_right < thr)
    say(f'rib failure (left or right DSC < {thr}): {side_fail.sum()}/{len(te)} ({100 * side_fail.mean():.1f}%); '
        f'DeepInspire misclassified among these: {(te.pred_di != y)[side_fail].sum()}, among the rest: {(te.pred_di != y)[~side_fail].sum()}')
d = te.rol_pred - te.rol_gt
say(f'ROL predicted minus GT (test): mean {d.mean():+.2f} points, SD {d.std():.2f}, 95% limits of agreement '
    f'{d.mean() - 1.96 * d.std():+.2f} to {d.mean() + 1.96 * d.std():+.2f}; same class at the cutoff: {(te.pred_di == te.pred_gt_rol).sum()}/{len(te)}')


## Segmentation (test, predicted vs GT at 512; distances in mm)
lung: DSC both 0.978±0.006, left 0.979, right 0.977 | HD95 median 4.2 mm (IQR 3.4–5.2) | ASSD median 1.3 mm (IQR 1.1–1.7) | empty predictions 0
rib: DSC both 0.753±0.235, left 0.708, right 0.793 | HD95 median 4.2 mm (IQR 2.9–24.2) | ASSD median 1.4 mm (IQR 0.9–4.0) | empty predictions 0
rib failure (left or right DSC < 0.3): 9/51 (17.6%); DeepInspire misclassified among these: 0, among the rest: 3
rib failure (left or right DSC < 0.5): 10/51 (19.6%); DeepInspire misclassified among these: 1, among the rest: 2
ROL predicted minus GT (test): mean -0.12 points, SD 5.40, 95% limits of agreement -10.71 to +10.47; same class at the cutoff: 48/51


Alternative cutoffs chosen on the training set.

In [13]:
# ---------------------------------------------------------------- alternative cutoffs
say('\n## Alternative cutoffs (chosen on training set, applied to test)')
rol_tr = tr.rol_pred.values; cands = np.unique(rol_tr); alt = [('Youden (current)', cut)]
for target in [0.95, 0.97, 1.0]:
    c = min(c for c in cands if metrics(y_tr, (rol_tr <= c).astype(int))['Sensitivity'] >= target)
    alt.append((f'training sensitivity >= {target:.2f}', c))
arow = []
for name, c in alt:
    mtr, mte = metrics(y_tr, (rol_tr <= c).astype(int)), metrics(y, (te.rol_pred.values <= c).astype(int))
    arow.append({'Rule': name, 'Cutoff %': round(c, 2), 'Train sens': round(mtr['Sensitivity'], 3), 'Train spec': round(mtr['Specificity'], 3),
                 'Test sens': round(mte['Sensitivity'], 3), 'Test spec': round(mte['Specificity'], 3),
                 'Test missed inadequate (FN)': int(((te.rol_pred.values > c) & (y == 1)).sum()),
                 'Test unnecessary repeats (FP)': int(((te.rol_pred.values <= c) & (y == 0)).sum())})
AT = pd.DataFrame(arow); AT.to_csv(os.path.join(O, 'table_alternative_cutoffs.csv'), index=False)
say(AT.to_string(index=False))


## Alternative cutoffs (chosen on training set, applied to test)
                        Rule  Cutoff %  Train sens  Train spec  Test sens  Test spec  Test missed inadequate (FN)  Test unnecessary repeats (FP)
            Youden (current)     83.92       0.986       1.000      0.889      0.970                            2                              1
training sensitivity >= 0.95     82.51       0.959       1.000      0.889      0.970                            2                              1
training sensitivity >= 0.97     82.54       0.973       1.000      0.889      0.970                            2                              1
training sensitivity >= 1.00     94.05       1.000       0.541      1.000      0.758                            0                              8


Full-resolution check.

In [14]:
# ---------------------------------------------------------------- resolution
say('\n## Full-resolution check (test)')
say(f"observer mask size matches native image size: observer1 {int(te.observer1_shape_ok.sum())}/{len(te)}, observer2 {int(te.observer2_shape_ok.sum())}/{len(te)}")
dn = te.rol_pred_native - te.rol_pred
say(f'ROL from masks upsampled to native size minus ROL at 512: mean {dn.mean():+.2f}, max |diff| {dn.abs().max():.2f} points; '
    f'class unchanged at cutoff: {((te.rol_pred_native <= cut) == (te.rol_pred <= cut)).sum()}/{len(te)}')
for o in ['observer1', 'observer2']:
    do = te[f'rol_{o}_native'] - te[f'rol_{o}_512']
    say(f'{o}: ROL at native minus ROL after resizing to 512: mean {do.mean():+.2f}, max |diff| {do.abs().max():.2f}; '
        f'class unchanged: {((te[f"rol_{o}_native"] <= cut) == (te[f"rol_{o}_512"] <= cut)).sum()}/{len(te)}')
    for s in ['rib', 'lung']:
        say(f'  {s} DSC predicted vs {o}: native {te[f"dsc_{s}_pred_vs_{o}_native"].mean():.3f}, at 512 {te[f"dsc_{s}_pred_vs_{o}_512"].mean():.3f}')


## Full-resolution check (test)
observer mask size matches native image size: observer1 51/51, observer2 51/51
ROL from masks upsampled to native size minus ROL at 512: mean +0.01, max |diff| 0.28 points; class unchanged at cutoff: 51/51
observer1: ROL at native minus ROL after resizing to 512: mean -0.01, max |diff| 0.21; class unchanged: 51/51
  rib DSC predicted vs observer1: native 0.744, at 512 0.743
  lung DSC predicted vs observer1: native 0.973, at 512 0.973
observer2: ROL at native minus ROL after resizing to 512: mean +0.02, max |diff| 0.40; class unchanged: 51/51
  rib DSC predicted vs observer2: native 0.694, at 512 0.700
  lung DSC predicted vs observer2: native 0.975, at 512 0.975


Table 1 (age and sex per subject), and save the summary.

In [15]:
# ---------------------------------------------------------------- Table 1 (per subject)
used = meta[~meta.image.isin(set(te_all.image) - set(te.image))]  # drop excluded test images, if any
t1 = []
for name, g in [('Total', used)] + [(n, used[used.set == s]) for n, s in
                                    [('Train', 'train'), ('Validation', 'validation'), ('Test', 'test')]]:
    p = g.drop_duplicates('subject_id')
    t1.append({'Group': name, 'Subjects': len(p), 'Age, median (range)': f'{p.age.median():g} ({p.age.min()}–{p.age.max()})',
               'Male': int((p.sex == 'M').sum()), 'Female': int((p.sex == 'F').sum()), 'Images': len(g),
               'Adequate': int((g.label == 'adequate').sum()), 'Inadequate': int((g.label == 'inadequate').sum())})
T1 = pd.DataFrame(t1); T1.to_csv(os.path.join(O, 'table1_per_subject.csv'), index=False)
say('\n## Table 1 (age and sex per subject)'); say(T1.to_string(index=False))

te_all.to_csv(os.path.join(O, 'per_image_test_with_flags.csv'), index=False)
open(os.path.join(O, 'stats_summary.txt'), 'w').write('\n'.join(out_lines) + '\n')
with open(os.path.join(O, 'cutoff.txt'), 'w') as f: f.write(repr(float(cut)))  # used by visualization.ipynb
print('\nresults written to', O)


## Table 1 (age and sex per subject)
     Group  Subjects Age, median (range)  Male  Female  Images  Adequate  Inadequate
     Total       166          62 (18–90)    83      83     223       119         104
     Train       109          62 (18–88)    51      58     148        74          74
Validation        20        63.5 (18–88)    11       9      24        12          12
      Test        40        60.5 (19–90)    21      19      51        33          18

results written to ../../results


## 3. Table 2: DSC per side for the automated segmentation and both annotators

Each annotator's full-resolution masks are resized to 512 × 512 (nearest neighbor) and compared with the GT masks. "Both" is the mean of left and right.

In [16]:
flags = pd.read_csv(os.path.join(OUT_DIR, 'per_image_test_with_flags.csv'), dtype={'number': str})
rows = []
for _, r in flags.iterrows():
    folder = 'not full' if r.y_inadequate else 'full'
    d = {'image': r.image, 'clean': not (r.subject_in_train or r.subject_in_valid),
         'model_rib_left': r.dsc_rib_left, 'model_rib_right': r.dsc_rib_right,
         'model_lung_left': r.dsc_lung_left, 'model_lung_right': r.dsc_lung_right}
    for part, sub in [('rib', 'Rib'), ('lung', 'Lung')]:
        g = labels_from_gt(os.path.join(DATA_DIR, 'test', folder, 'mask', part, f'{part}_' + r.image.replace('.png', '.tiff')))
        for o, name in [('observer1', 'annotator1'), ('observer2', 'annotator2')]:
            m = tifffile.imread(os.path.join(DATA_DIR, 'human_segment', f'mask by {o}', sub, r.number + '.tiff'))
            m = cv2.resize(m.astype(np.uint8), (512, 512), interpolation=cv2.INTER_NEAREST)
            d[f'{name}_{part}_left'], d[f'{name}_{part}_right'] = dsc(m == 1, g == 1), dsc(m == 2, g == 2)
    rows.append(d)
D = pd.DataFrame(rows)
out = []
subsets = [(f'test ({D.clean.sum()})', D.clean.values)]
if not D.clean.all():
    subsets.insert(0, (f'all ({len(D)})', np.ones(len(D), bool)))
for subset, sel in subsets:
    for part in ['lung', 'rib']:
        for side in ['left', 'right', 'both']:
            row = {'Test set': subset, 'Part': part, 'Side': side}
            for who in ['model', 'annotator1', 'annotator2']:
                l, rr = D.loc[sel, f'{who}_{part}_left'].mean(), D.loc[sel, f'{who}_{part}_right'].mean()
                row[who] = round({'left': l, 'right': rr, 'both': (l + rr) / 2}[side], 3)
            out.append(row)
T = pd.DataFrame(out)
T.to_csv(os.path.join(OUT_DIR, 'table2_dsc_by_side.csv'), index=False)
print(T.to_string(index=False))

 Test set Part  Side  model  annotator1  annotator2
test (51) lung  left  0.979       0.972       0.971
test (51) lung right  0.977       0.972       0.973
test (51) lung  both  0.978       0.972       0.972
test (51)  rib  left  0.708       0.884       0.854
test (51)  rib right  0.793       0.888       0.867
test (51)  rib  both  0.750       0.886       0.861
